In [37]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("C:/Users/Administrator/PythonProjects/abfluss_queich")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
from pathlib import Path

from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup, Tag
import pandas as pd

from utils.logger import logger
from jobs.temp.stations import STATIONS

In [4]:
TEMP_URL = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/10_minutes/air_temperature/now/"

In [51]:
def get_upload_time(html_tag: Tag) -> pd.Timestamp | None:

    tail = str(html_tag.next_sibling)
    
    if tail is None:
        return None
    
    parts = tail.strip().split()
    
    upload_time = pd.to_datetime(
        f"{parts[0]} {parts[1]}",
        dayfirst=True
    )
    
    return upload_time


def get_station_id(href: str) -> str:
    
    _, _, station_id, _ = href.split("_", maxsplit=3)
    
    return station_id


def fetch_temp_metadata(
    url: str | Path,
    station_ids: list[str],
    ) -> pd.DataFrame:
    try:
        response = requests.get(str(url), timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        
        
        rows = []

        for a in soup.find_all("a"):

            # Get filename
            href = str(a.get("href"))
            
            if not href:
                continue

            if not href.endswith(".zip"):
                continue
            
            if all(station not in href for station in station_ids):
                continue
            
            
            station_id = get_station_id(href=href)
            upload_time = get_upload_time(html_tag=a)
            
            if upload_time is None:
                continue
            
            
            rows.append({
                        "filename": href,
                        "station_id": station_id,
                        "datetime_upload": upload_time,
                    })

        return pd.DataFrame(rows)
    
    
    except requests.RequestException as e:
        logger.exception("Failed to fetch icon metadata from %s", url)
        raise
    
    
def download_radolan_file(
    root_url: str,
    file_name: str,
    output_dir: str
    ) -> None:
    
    file_url = urljoin(root_url, file_name)

    output_path = Path(output_dir) / file_name
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    if output_path.exists():
        logger.info("File %s yet exists, skip download", file_name)
        return
        
    try:
        logger.info("Downloading %s", file_name)    
        
        with requests.get(file_url, timeout=30, stream=True) as r:
                    
            r.raise_for_status()

            with open(output_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        
        logger.info("Saved file to %s", output_path)
        
        
    except requests.RequestException:
        logger.exception("Failed downloading file: %s", file_url)
        raise
    
    except OSError:
        logger.exception("Failed writing file: %s", output_path)
        raise

In [52]:
df = fetch_temp_metadata(url=TEMP_URL, station_ids=STATIONS)

In [53]:
df

,filename,station_id,datetime_upload
0,10minutenwerte_TU_00377_now.zip,00377,2026-05-31 20:20:00
1,10minutenwerte_TU_02486_now.zip,02486,2026-05-31 20:20:00
2,10minutenwerte_TU_03939_now.zip,03939,2026-05-31 20:20:01
3,10minutenwerte_TU_05426_now.zip,05426,2026-05-31 20:20:01


## RADOLAN NEW

In [ ]:
from pathlib import Path
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup, Tag
import pandas as pd

from configs.jobs_config import RADOLAN_OUTPUT_DIR, RADOLAN_DECOMPRESSED_DIR
from utils.logger import logger 

RADOLAN_URL = "https://opendata.dwd.de/climate_environment/CDC/grids_germany/hourly/radolan/recent/bin/"

RADOLAN_OUTPUT_DIR = "data/test/radolan/compressed"

RADOLAN_DECOMPRESSED_DIR = "data/test/radolan/decompressed"

def get_upload_time(html_tag: Tag) -> pd.Timestamp | None:

    tail = str(html_tag.next_sibling)
    
    if tail is None:
        return None
    
    parts = tail.strip().split()
    
    upload_time = pd.to_datetime(
        f"{parts[0]} {parts[1]}",
        dayfirst=True
    )
    
    return upload_time


def get_observation_date(href: str) -> pd.Timestamp:
    
    _, _, date, _ = href.split("-", maxsplit=3)
    
    return pd.to_datetime(date, format="%y%m%d%H%M")


def fetch_radolan_metadata(url: str | Path) -> pd.DataFrame:
    try:
        response = requests.get(str(url), timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        
        
        rows = []

        for a in soup.find_all("a"):

            # Get filename
            href = str(a.get("href"))
            
            if not href:
                continue

            if not href.endswith(".gz"):
                continue
            
            
            observation_date = get_observation_date(href=href)
            upload_time = get_upload_time(html_tag=a)
            
            if upload_time is None:
                continue
            
            # if upload_time < pd.Timestamp.now() - pd.DateOffset(years=1):
            if upload_time < pd.Timestamp.now() - pd.DateOffset(days=1):
                continue    
            
            
            rows.append({
                        "filename": href,
                        "datetime_observation": observation_date,
                        "datetime_upload": upload_time,
                    })

        return pd.DataFrame(rows)
        
        
    except requests.RequestException as e:
        logger.exception("Failed to fetch icon metadata from %s", url)
        raise
    

def get_local_observation_date(directory: str | Path):
    directory = Path(directory)
    
    observation_dates = set()
    
    for file in directory.glob("*.gz"):
        observation_date = get_observation_date(href=file.name)
        
        observation_dates.add(observation_date)
        
    return observation_dates


def download_radolan_file(
    root_url: str,
    file_name: str,
    output_dir: str
    ) -> None:
    
    file_url = urljoin(root_url, file_name)

    output_path = Path(output_dir) / file_name
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    if output_path.exists():
        logger.info("File %s yet exists, skip download", file_name)
        return
        
    try:
        logger.info("Downloading %s", file_name)    
        
        with requests.get(file_url, timeout=30, stream=True) as r:
                    
            r.raise_for_status()

            with open(output_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        
        logger.info("Saved file to %s", output_path)
        
        
    except requests.RequestException:
        logger.exception("Failed downloading file: %s", file_url)
        raise
    
    except OSError:
        logger.exception("Failed writing file: %s", output_path)
        raise
    
    
    
def fetch_radolan() -> None:
    
    df_remote = fetch_radolan_metadata(url=RADOLAN_URL)
    local_times = get_local_observation_date(directory=RADOLAN_OUTPUT_DIR)

    df_missing = df_remote[
        ~df_remote["datetime_observation"].isin(local_times)
    ]

    logger.info("Found %s new precip observation files", len(df_missing))
    
    files = 0
    for _, row in df_missing.iterrows():

        download_radolan_file(
            root_url=RADOLAN_URL,
            file_name=row["filename"],
            output_dir=RADOLAN_OUTPUT_DIR,
        )
        
        files += 1

    logger.info("Downloaded %s new precip observation files", files)


In [ ]:
df_remote = fetch_radolan_metadata(url=RADOLAN_URL)
local_times = get_local_observation_date(directory=RADOLAN_OUTPUT_DIR)

df_missing = df_remote[
    ~df_remote["datetime_observation"].isin(local_times)
]

logger.info("Found %s new precip observation files", len(df_missing))

files = 0
for _, row in df_missing.iterrows():

    download_radolan_file(
        root_url=RADOLAN_URL,
        file_name=row["filename"],
        output_dir=RADOLAN_OUTPUT_DIR,
    )
    
    files += 1

logger.info("Downloaded %s new precip observation files", files)


2026-05-31 23:10:52,394 | 3242363423.py | INFO | Found 0 new precip observation files
2026-05-31 23:10:52,394 | 3242363423.py | INFO | Downloaded 0 new precip observation files


In [55]:
fetch_radolan()

KeyError: 'datetime_observation'